# TikTok: reclamación u opinión, y quién hizo el trabajo

**Curso 5 del certificado, proyecto de modelos basados en árboles.**

Este cuaderno construye el clasificador que el Curso 2 anunciaba, y después dedica el resto
de su espacio a demostrar que **el algoritmo no aportó casi nada**.

**Lo que espero, escrito antes de ajustar nada:** el Curso 2 ya midió que una reclamación se
ve 101 veces más que una opinión, con los dos repartos casi sin solaparse. Con esa señal
cualquier modelo sale casi perfecto, así que un 99 % no será un éxito: será una confirmación.

Por eso hay un control fijado de antemano.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "projects" / "curso5").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "projects"))
sys.path.insert(0, str(ROOT / "projects" / "curso5" / "tiktok" / "02_scripts"))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier

from tiktok_trees import FEATURES, LONE, SEED, load, split
from common import Results, dataset

pd.set_option("display.width", 120)
print("listo")

listo


## 1. Los datos, y una diferencia con el Curso 4 que conviene mirar

Aquel proyecto predecía si la cuenta estaba verificada: 93,6 % contra 6,4 %, así que tuvo
que equilibrar las clases. Aquí el objetivo es otro y está repartido casi por mitades.

In [2]:
results = Results("tiktok", "TikTok, cuaderno del Curso 5", "curso5")
df = load(results)
print()
print(df.claim_status.value_counts(normalize=True).round(4).to_string())


1. Los datos, y el objetivo que cambia respecto al Curso 4
  filas en el CSV: 19382
  filas incompletas, las mismas 298 del Curso 2: 298
  filas completas: 19084
  porcentaje de reclamaciones: 50.3
  ¿hace falta equilibrar? En el Curso 4 sí, aquí no: False
  cuántas veces más se ve una reclamación, el hallazgo del Curso 2: 101.1
  la opinión más vista de todo el archivo: 9998
  la reclamación menos vista: 1049

claim_status
claim      0.5035
opinion    0.4965


**50,3 % contra 49,7 %.** Mismos datos que el Curso 4 y el problema opuesto: el desbalance
no es una propiedad del archivo, es una propiedad de la pregunta que le haces.

## 2. El control, fijado antes de tocar nada

Un árbol de **una sola pregunta sobre una sola columna**. Es el modelo más tonto que se
puede escribir, y todo lo demás tendrá que ganarle.

Se decide ahora y no después, porque elegir la referencia una vez que ya has visto tu
resultado es cómo un proyecto se convence a sí mismo de ser impresionante.

In [3]:
train, validation, test = split(df, results)

stump = DecisionTreeClassifier(max_depth=1, random_state=SEED).fit(
    train[[LONE]], train.is_claim)
cut = stump.tree_.threshold[0]

print(f"la regla entera: si el video pasa de {cut:,.0f} visualizaciones, es reclamacion")
print(f"exactitud en validacion: "
      f"{accuracy_score(validation.is_claim, stump.predict(validation[[LONE]])):.4f}")


2. Partición en tres, con la prueba apartada desde el principio
  entrenamiento 11450   validación 3817   prueba 3817
la regla entera: si el video pasa de 10,032 visualizaciones, es reclamacion
exactitud en validacion: 0.9961


**99,61 % con una sola pregunta.** Antes de entrenar nada más, ya sabemos que el techo está
altísimo y que va a costar mucho justificar cualquier cosa más complicada.

## 3. Por qué esa pregunta funciona tan bien

Aquí está el número que cierra el asunto.

In [4]:
views = df.groupby("claim_status").video_view_count
print(f"opinion mas vista de todo el archivo: {views.max()['opinion']:,.0f}")
print(f"el corte que eligio el modelo:        {cut:,.0f}")
print(f"reclamacion menos vista:              {views.min()['claim']:,.0f}")
print()
print(f"cociente de medias: {views.mean()['claim'] / views.mean()['opinion']:.1f} veces")

opinion mas vista de todo el archivo: 9,998
el corte que eligio el modelo:        10,032
reclamacion menos vista:              1,049

cociente de medias: 101.1 veces


**La opinión más vista de todo el archivo tiene 9.998 visualizaciones y el corte cae en
10.031.** Las dos clases están separadas por una línea.

Y aquí llega la parte incómoda: **un hueco así no existe en una plataforma real.** Ningún
vídeo de opinión de TikTok tiene un techo de diez mil visualizaciones. Aparece cuando alguien
genera dos poblaciones con rangos distintos, y **estos datos son sintéticos**.

![Las dos clases y el corte](03_figures/01_el_corte.png)

## 4. Los modelos de verdad, con las nueve variables

Medidos en validación, sin tocar la prueba.

In [5]:
X, y = train[FEATURES], train.is_claim
Xv, yv = validation[FEATURES], validation.is_claim

for name, model in {
    "logistica": LogisticRegression(max_iter=3000),
    "bosque":    RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
}.items():
    model.fit(X, y)
    print(f"{name:11} AUC {roc_auc_score(yv, model.predict_proba(Xv)[:, 1]):.4f}   "
          f"exactitud {accuracy_score(yv, model.predict(Xv)):.4f}")

logistica   AUC 0.9968   exactitud 0.9932
bosque      AUC 0.9975   exactitud 0.9963


## 5. La prueba, una vez, con el control dentro

Esta es la tabla del proyecto.

In [6]:
forest = RandomForestClassifier(n_estimators=300, max_features="sqrt",
                                min_samples_leaf=5, random_state=SEED, n_jobs=-1).fit(X, y)
logistic = LogisticRegression(max_iter=3000).fit(X, y)

rows = [("una pregunta, 1 columna", stump, [LONE]),
        ("logistica, 9 variables", logistic, FEATURES),
        ("bosque ajustado, 9 variables", forest, FEATURES)]
for name, model, columns in rows:
    p = model.predict_proba(test[columns])[:, 1]
    print(f"{name:30} exactitud {accuracy_score(test.is_claim, model.predict(test[columns])):.4f}"
          f"   AUC {roc_auc_score(test.is_claim, p):.4f}")

una pregunta, 1 columna        exactitud 0.9958   AUC 0.9958
logistica, 9 variables         exactitud 0.9940   AUC 0.9973
bosque ajustado, 9 variables   exactitud 0.9958   AUC 0.9986


**Nueve variables y trescientos árboles aportan 0,0000 de exactitud sobre una sola
pregunta.** La logística, con las nueve, saca menos.

El AUC sí mejora, de 0,9958 a 0,9986, y eso es real: el bosque ordena mejor los casos
dudosos. Pero a la hora de decidir, que es lo que se despliega, no mejora nada.

![El control contra los contendientes](03_figures/02_una_pregunta.png)

## 6. Los únicos vídeos difíciles del archivo

La regla de una línea se equivoca con 65 vídeos, y los 65 son del mismo tipo.

In [7]:
wrong = ((train.claim_status.eq("opinion") & train[LONE].gt(cut)) |
         (train.claim_status.eq("claim") & train[LONE].le(cut)))
print(f"videos que la regla no acierta: {int(wrong.sum())} de {len(train)}")
print()
print(train[wrong].claim_status.value_counts().to_string())
print()
print(train.loc[wrong, LONE].describe().round(0).to_string())

videos que la regla no acierta: 65 de 11450

claim_status
claim    65

count      65.0
mean     5632.0
std      2363.0
min      1049.0
25%      3616.0
50%      5449.0
75%      7743.0
max      9754.0


Los 65 son **reclamaciones que casi nadie vio**. Son los únicos casos con algo que aprender,
y la diferencia entre 0,9958 y 0,9986 de AUC se juega entera ahí dentro.

![Los 65 vídeos difíciles](03_figures/04_excepciones.png)

![La importancia de las variables](03_figures/03_importancia.png)

## 7. Qué se decide con esto

**No desplegar esto como clasificador de contenido.** Lo que aprendió es un umbral de
visualizaciones que solo existe en el archivo de prácticas.

**Y quedarse con lo que sí enseña:** que un control tonto fijado de antemano es lo que
convierte un 99 % en información. Sin él, este cuaderno habría terminado en la primera tabla,
felicitándose.